In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_train.csv
/kaggle/input/competitions/titanic-prediction-scc-study-group/sample submission.csv
/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_test.csv


他code学習①

26/5/25-5/31

#  Titanic competition w/ TensorFlow Decision Forests

In [2]:
import numpy as np
import pandas as pd
import os

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print(f"Found TF-DF {tfdf.__version__}")

2026-05-30 17:09:01.028082: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780160941.226464      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780160941.276853      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780160941.720077      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780160941.720127      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780160941.720131      16 computation_placer.cc:177] computation placer alr

Found TF-DF 1.12.0



・import tensorflow as tf  
→ TensorFlow（深層学習フレームワーク）を読み込む

・import tensorflow_decision_forests as tfdf  
→ 決定木系モデル（Random Forest / Gradient Boosted Trees）を TensorFlow で使えるようにする

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_train.csv")
serving_df = pd.read_csv("/kaggle/input/competitions/titanic-prediction-scc-study-group/titanic_test.csv")

# 前処理方法、名前を分割、チケットを数字と文字列に分割

In [4]:
def preprocess(df):
	df = df.copy()
	
	def normalize_name(x):
		return " ".join([v.strip(",()[].\"'") for v in x.split(" ")]) #joinに複数の要素はリストにして渡す

	def ticket_number(x):
		return x.split(" ")[-1]

	def ticket_item(x):
		items = x.split(" ")
		if len(items) == 1:
			return "NONE"
		return "_".join(items[0:-1])
		
	df["Name"] = df["Name"].apply(normalize_name)
	df["Ticket_number"] = df["Ticket"].apply(ticket_number)
	df["Ticket_item"] = df["Ticket"].apply(ticket_item)
	return df
	
preprocessed_train_df = preprocess(train_df)
preprocessed_serving_df = preprocess(serving_df)

・stripは、左右両端の「前後の不要文字を削る」

# 不要な特徴量の削除　+　後ほど使う
純粋に予測に使いたい特徴量のリスト作成

In [5]:
input_features = list(preprocessed_train_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
input_features.remove("Survived")

print(f"Input features: {input_features}")

Input features: ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked', 'Ticket_number', 'Ticket_item']


● remove("Ticket")
→ Ticket はモデルに不要なので削除  
（文字列のままでは意味がない、ノイズになる）

● remove("PassengerId")
→ PassengerId も予測に関係ないので削除  
（ただの識別子）

● remove("Survived")
→ Survived は 目的変数（y） なので、
　特徴量（X）からは除外する必要がある

# pandasデータ→TensorFlowDatasetへの変換+「名前」をテキスト処理(トークン化)

・TensorFlowのモデルは、PandasのDataFrameをそのまま直接読み込んで学習することができません。TensorFlow専用の最適化されたデータ構造（tf.data.Dataset）に変換する必要があります。

・テキストをAIが扱いやすくするため: 「Braund, Mr. Owen Harris」という氏名の文字列をそのまま入れるより、「Braund」「Mr.」「Owen」「Harris」と単語ごとに分割（トークン化）してあげた方が、AI（特に決定木モデル）は「"Mr." が含まれる人は生存率が低い」といった一貫したルールを見つけやすくなるから

・TensorFlowテンソルフローはGoogle製のディープラーニング用プラットフォームである
・すべてのデータは「Tensor（多次元配列・行列）」として高速計算される
・初心者がコードを書くときは、TensorFlow内蔵の優しい機能「Keras（tf.keras）」を使う
・研究志向のPyTorchに対し、TensorFlowは実務の本番システムへの導入（デプロイ）に超強い

流れ
[Pandas DataFrame]
       │
       ▼ (① pd_dataframe_to_tf_dataset)
[TensorFlow Dataset (tf.data.Dataset)]
       │
       ▼ (② .map(tokenize_names))
[テキストが単語ごとに分割されたDataset]（モデルへの入力準備完了）

In [6]:
def tokenize_names(features, labels=None):
	features["Name"] = tf.strings.split(features["Name"])
	return features,labels
	
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_train_df,label="Survived").map(tokenize_names)

serving_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_serving_df).map(tokenize_names)

2026-05-30 17:09:18.946309: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


・label="Survived"
訓練データ（train_ds）のどれが正解ラベル（目的変数）かを指定しています。これにより、データセットが「特徴量」と「正解（Survived）」のペアに自動で分割されます。テストデータ（serving_ds）には正解がないので、この引数は指定していません。

・TF-DFの強み（裏側の挙動）：
通常のTensorFlow（ディープラーニング）だと、文字列やカテゴリ変数は手動でワンホットエンコーディング（数値化）する必要があります。しかし、TF-DFは文字列や数値、欠損値をそのままネイティブに受け付けることができるため、ここでは面倒なエンコーディングをせず、ただ型を変換するだけで済んでいます。

・TF‑DF の推奨前処理
TensorFlow Decision Forests の公式推奨：
カテゴリ → そのまま文字列で渡す
数値 → そのまま
欠損値 → 自動処理
One‑Hot → 不要
Label Encoding → 不要
factorize → 不要（むしろ使わない方が良い）

TF‑DF は 前処理を最小限にする設計。・

# モデル作成　+　学習　＋　予測「勾配ブースティング木(Gradient Boosted Trees)」

In [7]:
model = tfdf.keras.GradientBoostedTreesModel(
	verbose = 0,
	features = [tfdf.keras.FeatureUsage(name=n) for n in input_features],
	exclude_non_specified_features = True,
	random_seed = 0,
)

model.fit(train_ds) #TensorFlow スタイル（model.fit(dataset)）
	
self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy:{self_evaluation.accuracy} Loss:{self_evaluation.loss}")

W0000 00:00:1780160959.213226      16 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1780160959.213874      16 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1780160959.213887      16 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
I0000 00:00:1780160963.033455      16 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1780160963.033491      16 kernel.cc:783] Collect training examples
I0000 00:00:1780160963.033503      16 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
column_guides {
  column_name_pattern: "^Pclass$"
}
column_guides {
  column_name_pattern: "^Name$"
}
column_guides {
  column_name_pattern: "^Sex$"
}
column_guides {
  column_name_pattern: "^Age$"
}
column_guide

Accuracy:0.8840579986572266 Loss:0.5192124843597412


・[input_features]ではダメな理由
→「モデル（TF-DF）がただの『文字列のリスト(input_features)』では受け付けてくれず、専用の『設定オブジェクト（FeatureUsage型）のリスト』を要求する仕様だから」
# 変換後のリストの中身のイメージ（モデルが喜ぶ形式）
[
    FeatureUsage(name="Pclass", モード=自動, 型=自動, ...),
    FeatureUsage(name="Sex", モード=自動, 型=自動, ...),
    FeatureUsage(name="Age", モード=自動, 型=自動, ...),
    FeatureUsage(name="Ticket_number", モード=自動, 型=自動, ...)
]

・FeatureUsage は、単に「この列を使う」と指定するだけでなく、「その列をモデル内でどう特別扱いするか」という細かいオプション（設定）を書き込めるノートのような役割

例；
データの型を強制する（よく使う！）
「Pclass（客室等級）は 1, 2, 3 という数値だけど、計算用ではなく『カテゴリ（グループ）』として扱ってね」と指示したいとき。
tfdf.keras.FeatureUsage(name="Pclass", semantic=tfdf.keras.FeatureSemantic.CATEGORICAL)

特定の列だけ特別扱いする
「この列には欠損値が多いから別の方法で補完して」とか「この列は重要だから特別に重みをつけて学習して」といった、アルゴリズム側への細かい注文をここに書き込めます。

・FeatureUsage(name=n)　ここのnameは"Name"列とは違うから勘違い注意！

・exclude_non_specified_features=True
「リストにない列（PassengerId など）は完全に無視してね」という念押しです。

・なぜmodel.fit(train_ds)今回は train_ds 1つでいいのか？
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    preprocessed_train_df, 
    label="Survived" # 👈 ここがポイント！
).map(tokenize_names)

TensorFlowは内部でデータを次のような構造のペア（タプル）に自動で組み替えています
train_ds の中身の構造： ( features, labels )
features ＝ Survived 以外の特徴量（X）
labels ＝ Survived のデータ（y）

・今回使っている TF-DF（勾配ブースティング木） というライブラリは超優秀で、model.fit() をしている裏側で、人間が指示しなくても、自動でデータの一部を「実力テスト用」としてキープしています。
.make_inspector()：モデルの内部をのぞき込む「検査官（インスペクター）」を呼び出す
.evaluation()：その検査官に、キープしておいたデータでの「模擬試験の結果」を報告させる
・通常の機械学習（LightGBMなど）では、モデルの「実力」を測るために、人間がわざわざデータを「訓練用」と「検証用（テスト用）」に手動で分割して、検証用データを使って採点

・Loss（損失）とは？（簡潔にいうと「やらかし度」）
「この数値を 0.7 ➡ 0.5 ➡ 0.3 とゼロに近づけていくこと」が、Kaggleでスコアを上げる（＝モデルの迷いを無くし、確実な予測に磨き上げる）ための直接の目標
自信満々に間違うor当たっても自信はないなどの予測に対する実力ではない部分の正解が含まれるとき。


# **ハイパーパラメータ(設定値)をいじって精度向上を目指す**

・このパラメータ構成は、「時間をかけてでも、特徴量同士の複雑な関係を斜め分割（Oblique）で網羅し、小さな学習率でじっくり2000本の木を育てる」という非常に攻めた（表現力の高い）設定

In [8]:
model = tfdf.keras.GradientBoostedTreesModel(
	#①基本設定
	verbose = 0,
	features = [tfdf.keras.FeatureUsage(name=n) for n in input_features],
	exclude_non_specified_features = True,

	#②木の構造と学習に関するパラメータ
	min_examples = 1,
	categorical_algorithm = "RANDOM",
	shrinkage = 0.05,
	num_trees = 2000,
	random_seed = 0,
	
	#③軸を斜めに切る特殊な分割(Oblique Splits)
	split_axis = "SPARSE_OBLIQUE",
	sparse_oblique_normalization = "MIN_MAX",
	sparse_oblique_num_projections_exponent = 2.0,
)

W0000 00:00:1780160964.791920      16 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1780160964.791956      16 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1780160964.791961      16 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


②木の構造と学習に関するパラメータ
・min_examples=1・
	葉ノード（木の末端）に最低限必要なサンプル数を 1 に設定しています（デフォルトは通常 5 以上）。

	効果: データを限界まで細かく分類しようとするため、表現力が上がりますが、過学習（ノイズまで学習してしまうこと）のリスクが高まります。

・ategorical_algorithm="RANDOM"
	カテゴリ変数（文字列データなど）を分岐させる際、ランダムにグループを分けて最適な分割を探します。高速かつ過学習を防ぐ効果があります。

・shrinkage=0.05
	いわゆる学習率（Learning Rate）です。1つの木が予測した結果をどれくらい次の木に引き継ぐかを調整します。デフォルト（通常 0.1）より小さくすることで、慎重にじわじわと学習を進めます。

・num_trees=2000
	作成する決定木の数を 2000 本に増やしています。shrinkage を小さくした分、木の数を多くして補うアプローチです。


③軸を斜めに切る特殊な分割（Oblique Splits）

・split_axis="SPARSE_OBLIQUE"
	いくつかの特徴量をランダムに選び、それらを線形結合（足し算や引き算）して斜めの分岐条件を作ります（例：「$0.5 \times \text{年齢} + 0.8 \times \text{運賃} > 50$」のような条件）。

	効果: 特徴量同士の相互作用（相乗効果）を捉えやすくなり、タイタニックのような複雑なデータで高い表現力を発揮します。

・sparse_oblique_normalization="MIN_MAX"
	斜めの境界線を作る際、スケール（単位）の異なる特徴量（例：桁が小さい「乗船数」と桁が大きい「運賃」）をフェアに扱うために、自動でMin-Max正規化（0〜1の範囲にスケーリング）を行います。

・sparse_oblique_num_projections_exponent=2.0
	斜めの分割候補をいくつ作成するかを決める係数です。数値を大きくするほど、より多くの複雑な斜めの分割パターンをテストするため精度が向上しやすくなりますが、計算時間は長くなります。


# モデルの学習と評価

In [9]:
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print({f"Accuracy:{self_evaluation.accuracy} Loss:{self_evaluation.loss}"})

I0000 00:00:1780160965.109794      16 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1780160965.109844      16 kernel.cc:783] Collect training examples
I0000 00:00:1780160965.109855      16 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
column_guides {
  column_name_pattern: "^Pclass$"
}
column_guides {
  column_name_pattern: "^Name$"
}
column_guides {
  column_name_pattern: "^Sex$"
}
column_guides {
  column_name_pattern: "^Age$"
}
column_guides {
  column_name_pattern: "^SibSp$"
}
column_guides {
  column_name_pattern: "^Parch$"
}
column_guides {
  column_name_pattern: "^Fare$"
}
column_guides {
  column_name_pattern: "^Cabin$"
}
column_guides {
  column_name_pattern: "^Embarked$"
}
column_guides {
  column_name_pattern: "^Ticket_number$"
}
column_guides {
  column_name_pattern: "^Ticket_item$"
}
default_column_guide {
  categorial {
    max_vocab_

{'Accuracy:0.8840579986572266 Loss:0.6649019718170166'}


●評価

結構悪い。改善案として

1位：min_examples を引き上げる（即効性：★★★★★）
対策: min_examples=1 を min_examples=5〜15 程度に戻します。

2位：斜め分割を一度やめて「通常分割」にする（重要度：★★★★☆）
対策: split_axis="SPARSE_OBLIQUE" をコメントアウト（または "AXIS_ALIGNED" に変更）する。
理由: タイタニックのようなサンプル数が少ない（約890件）データでは、複雑な斜め分割は過学習の温床になります。まずは「直角な分割」で土台を固めるのがセオリーです。

# model.summary()
学習したモデルの構造や、モデルが「どの特徴量(列)を重要と判断したか(重要度=Variable Importance)」を詳しく確認するための最も強力なコマンド。

In [10]:
model.summary()

Model: "gradient_boosted_trees_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1 (1.00 Byte)
Trainable params: 0 (0.00 Byte)
Non-trainable params: 1 (1.00 Byte)
_________________________________________________________________
Type: "GRADIENT_BOOSTED_TREES"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (11):
	Age
	Cabin
	Embarked
	Fare
	Name
	Parch
	Pclass
	Sex
	SibSp
	Ticket_item
	Ticket_number

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1.           "Sex"  0.607811 ################
    2.          "Fare"  0.400008 ########
    3.           "Age"  0.386790 #######
    4.        "Pclass"  0.369291 #######
    5.         "SibSp"  0.297652 ####
    6.         "Parch"  0.245603 ##
    7.          "Name"  0.201211 
    8.   "Ticket_item"  0.185668 
    9.      "Embarked"  0.176387 
   10. "Ticket_number"  0.175965 

Variable Importance: NUM_AS_ROOT:
    1.    "Se

①SUM_SCORE（総合的な貢献度）：性別と数値データの三つ巴
モデル全体の予測精度にどれだけ直接貢献したかのトータルスコア

1位Sex (460.49) ################
2位Age (355.96) ############
3位Fare (292.87) ##########
4位Name (108.54) ###

②NUM_AS_ROOT（根ノードに選ばれた回数）：最強の切り札
33本ある決定木のうち、一番最初の分岐（最もデータを綺麗に二等分できる条件）に選ばれた回数
1位Sex: 28回
2位Name: 5回

現場の評価（危険度MAX）: 33本中28本で Sex が最初に使われており、文句なしの主役です。しかし、ゴミ特徴量であるはずの Name が5回も根っこ（一番上）に選ばれているのは異常事態です。モデルが完全に名前に依存し始めています。

③NUM_NODES（分岐に使われた総回数）：細かく刻まれたデータ
木の中で、その特徴量が何回分岐の条件として登場したかです。

Age: 406回
Fare: 290回
Name: 44回

現場の評価: Age（年齢）や Fare（運賃）は「10歳未満」「20歳以上」「運賃が50ドル以上」というように、数値を細かく刻んで何度も分岐に使えるため、回数が跳ね上がります。これは正常な挙動です。

●ログの奥底に眠る「過学習（丸暗記）」の動かぬ証拠
2つの過学習のサインあり

①葉ノードのデータ数が「1」になっている（末期状態）
Number of training obs by leaf:
[ 1, 22) 731 78.77%
Min: 1 Max: 438

木の末端（リーフ）に、乗客が 「1人」 しか残っていない部屋が大量に作られています（全リーフの約78%が、22人未満の激狭な部屋になっています）。これはただの過去問の丸暗記状態。
つまり1+2=3のやり方を覚えるんじゃなくて3という結果を覚えるような再現性のほぼない無意味な状態。2+3=？になっただけで予測が未熟になる

②学習ログ（Training logs）の逆転現象
モデルの成長記録です。ここを見ると、どこでモデルが力尽きたかが分かります。

》5回目（Iter:5）
練習問題（train）の正解率: 80.8% / 模擬試験（valid）の正解率: 77.1%
評価: 非常に健全。バランスよく学習できています。

》26回目（Iter:26）
練習問題: 92.6% / 模擬試験: 79.3%
評価: ここがピークです。

》56回目（最後の方）
練習問題: 93.8%（さらに上昇！） / 模擬試験: 77.1%（低下！）
valid-loss（損失）: 1.01 ➔ 1.03 へ悪化

現場のリアルな意見: 「56回目まで進むと、練習問題は93%も解けてるのに、本番を想定した模擬試験（valid）ではスコアが落ちとる。完全に途中で伸び悩んで、無駄に丸暗記に走った証拠やな」

# 提出ファイル作成処理

[1. 予測の出力と二値化] (prediction_to_kaggle_format) ──> 
[2. CSVファイルへの保存] ──>(make_submission) 
[3. 出力ファイルの確認] (!head)   

In [11]:
def prediction_to_kaggle_format(model, threshold=0.5):
	proba_survive = model.predict(serving_ds,verbose=0)[:,0]
	return pd.DataFrame({
		"PassengerId":serving_df["PassengerId"],
		"Survived":(proba_survive >= threshold).astype(int)
	})

def make_submission(kaggle_predictions):
	path = "/kaggle/working/submission.csv"
	kaggle_predictions.to_csv(path, index=False)
	print(f"Submission exported to {path}")

	
kaggle_predictions = prediction_to_kaggle_format(model)
make_submission(kaggle_predictions)
!head /kaggle/working/submission.csv

Submission exported to /kaggle/working/submission.csv
PassengerId,Survived
566,0
161,0
554,0
861,0
242,1
560,0
388,1
537,1
699,0


① prediction_to_kaggle_format(model, threshold=0.5)
モデルの「確率出力」を、Kaggleが指定する「0か1のフラグ」に変換し、提出用フォーマットのデータフレームを作成する関数
	》model.predict(serving_ds,verbose=0)[:,0]
	・モデルにテストデータ（serving_ds）を流し込み、
	各乗客が「生存する確率（0.0～1.0）」を予測しています。
	・[:,0]を指定して、多次元配列から1次元目の
	「生存確率」の列だけを抽出しています。

	》"Survived": (proba_survive >= threshold).astype(int)
	・ここが非常に重要です。モデルが出力した確率はそのままでは提出できません
	（Kaggleのタイタニックお題では 0 または 1 を求められるため）。
	・設定されたしきい値（threshold=0.5、デフォルトは50%）
	を基準に、確率が0.5以上なら True（生存）、0.5未満なら False（死亡） というブール値に変換します。
	・最後に .astype(int) を通すことで、True を 1、False を 0 の数値に変換しています。

② make_submission(kaggle_predictions)
作成したデータフレームを、実際のファイル（CSV形式）としてKaggleの作業環境に書き出す関数です。

	》path="/kaggle/working/submission.csv"
	・Kaggle Notebook環境において、
	ユーザーがファイルを書き出すことができる標準的な
	ディレクトリ（/kaggle/working/）とファイル名を指定しています。

	》kaggle_predictions.to_csv(path, index=False)
	・PandasのデータフレームをCSVファイルに変換して保存します。
	・index=False の設定がポイントです。
	これを忘れると、Pandasが自動で割り振る行番号
	（0, 1, 2...）が余計な列としてCSVに含まれてしまい、
	Kaggle側でエラー（フォーマット不正）になってしまいます。

③実行部分と確認（!head）
最後に関数を実行し、作成されたファイルの中身を先頭から数行だけ覗き見ています。

	》!head /kaggle/working/submission.csv
	・!head はLinuxのコマンドで、
	指定されたファイルの先頭10行を表示する命令です。
	出力結果を見ると、1行目にヘッダー（PassengerId,Survived）
	があり、2行目以降に正しく 892,0のようにカンマ区切りでIDと0/1の予測値が
	並んでいることが確認できます。フォーマットとして完璧です。

・問、なぜ、model.predict(serving_ds, verbose=0)[:,0]の1次元目に「生存確率」の列が来ているとわかるの？

→Python（NumPy/TensorFlow）の少しややこしい、でも大切なルール
Pandasのデータフレームに新しく列を追加するとき、データは「ただの1列のリスト（1次元配列）」である必要があります。しかし、model.predict() が返してくるのは、上記のように無駄にカッコが二重になった「2次元配列」です。

スライス記号を使うことで、
例、# [:, 0] をすることで、二重のカッコが外れて綺麗な1次元配列になる
	[0.12, 0.85, 0.43]


# 最適な設定値(ハイパーパラメータ)をAIに自動で探させるコード
勘や手作業での設定値（パラメータ）調整を卒業し、「自動探索（チューニング）」によってモデルの性能を限界まで引き上げる仕組

☑何を覚えるべきか（優先順位）
1、最優先： tuner.choice("名前", [候補リスト]) の書き方。これだけで自動探索は始められます。

2、次に： 過学習を防ぐ min_examples や max_depth は必ずチューニングの候補に入れること。

3、余裕があれば： merge=True を使った条件付きの探索（高度なテクニックですが、Kaggle上位を狙うなら必須です）

In [12]:
#①チューナーの初期化
tuner = tfdf.tuner.RandomSearch(num_trials=1000)

#②基本パラメータの設定
tuner.choice("min_examples",[2,5,7,10])
tuner.choice("categorical_algorithm",["CART","RANDOM"])

#③条件付き探索：木の成長戦略（LOCAL）
local_search_space = tuner.choice("growing_strategy",["LOCAL"])

local_search_space.choice("max_depth", [3,4,5,6,8])

#④条件付き探索：木の成長戦略（BEST_FIRST_GLOBAL）
global_search_space = tuner.choice("growing_strategy",["BEST_FIRST_GLOBAL"],merge=True)

#⑤共通パラメータの設定（ブースティング関連）
tuner.choice("shrinkage",[0.02, 0.05, 0.10, 0.15])
tuner.choice("num_candidate_attributes_ratio", [0.2, 0.5, 0.9, 1.0])

#⑥分割軸の設定と条件付き探索（斜め分割）
tuner.choice("split_axis", ["AXIS_ALIGNED"])
oblique_space =tuner.choice("split_axis", ["SPARSE_OBLIQUE"],merge=True)

oblique_space.choice("sparse_oblique_normalization", ["NONE","STANDARD_DEVIATION","MIN_MAX"])
oblique_space.choice("sparse_oblique_weights", ["BINARY", "CONTINUOUS"])
oblique_space.choice("sparse_oblique_num_projections_exponent", [1.0, 1.5])

#⑦モデルの定義と訓練
tuned_model = tfdf.keras.GradientBoostedTreesModel(tuner=tuner)

tuned_model.fit(train_ds, verbose=0)

#⑧評価と結果出力
tuned_self_evaluation = tuned_model.make_inspector().evaluation()

print(f"Accuracy:{tuned_self_evaluation.accuracy} Loss:{tuned_self_evaluation.loss}")

Use /tmp/tmpoeiktxs7 as temporary training directory


W0000 00:00:1780160966.811725      16 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1780160966.811772      16 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1780160966.811779      16 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
I0000 00:00:1780160967.101927      16 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1780160967.101967      16 kernel.cc:783] Collect training examples
I0000 00:00:1780160967.101979      16 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false


Accuracy:0.8939393758773804 Loss:0.5940573215484619


●実務・Kaggleでの重要度と頻出度
Kaggleでの頻出度: ★☆☆☆☆ (低い)

KaggleではTF-DFよりも、LightGBM、XGBoost、CatBoostが圧倒的にシェアを占めています。TF-DFを使うケースは、TensorFlowのパイプラインにどうしても組み込みたい場合がメインです。

実務での頻出度: ★★☆☆☆ (やや低い)

すでにTensorFlowでディープラーニングのシステムが組まれており、そこに決定木ベースのモデルをサクッと統合したい現場では重宝されます。

初心者の学習優先順位: ★☆☆☆☆ (後回しでOK)

まずは LightGBM や Scikit-learn の GridSearchCV / RandomizedSearchCV を学ぶ方が、情報量も多く実務に直結します。